In [22]:
!pip install fvcore av pytorchvideo

In [23]:
import os
import cv2
import torch
import torch.nn as nn
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import *
from tensorflow.keras.models import Model
from google.colab import drive
from google.colab import files
from torchvision.transforms import Compose, Resize, CenterCrop

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using Device: {DEVICE}")

print("\nLoading Model 1 (Water Detection)...")
model_water = torch.hub.load("facebookresearch/pytorchvideo", "slow_r50", pretrained=False)
model_water.blocks[5].proj = nn.Linear(in_features=2048, out_features=2, bias=True)

model_1_path = "/content/drive/MyDrive/best_drone_water_model-v2.pth"
model_water.load_state_dict(torch.load(model_1_path, map_location=DEVICE))
model_water.to(DEVICE)
model_water.eval()
print("Model 1 loaded successfully!")

print("\nLoading Model 2 (Pollution Detection)...")
model_2_path = "/content/drive/MyDrive/ConvNeXt_Water_Best.h5"

def build_convnext_final(input_shape=(224, 224, 3), num_classes=6):
    inputs = tf.keras.Input(shape=input_shape)

    base_model = tf.keras.applications.ConvNeXtTiny(
        include_top=False,
        weights=None,
        input_tensor=inputs
    )

    x = GlobalAveragePooling2D()(base_model.output)
    x = BatchNormalization()(x)

    x = Dense(256, activation='relu')(x)
    x = Dropout(0.5)(x)

    outputs = Dense(num_classes, activation='softmax')(x)
    model = Model(inputs, outputs)
    return model

try:
    model_pollution = build_convnext_final()
    model_pollution.load_weights(model_2_path)
    print("Model 2 (ConvNeXt) loaded successfully!")
except Exception as e:
    print(f"Direct Load failed, attempting safe fallback compilation...")
    try:
        model_pollution = tf.keras.models.load_model(model_2_path, compile=False, safe_mode=False)
        print("Model 2 loaded via final security bypass!")
    except Exception as e2:
        print(f"Error loading Model 2: {e2}")

water_transform = Compose([
    Resize((256, 256)),
    CenterCrop((224, 224))
])

Using Device: cuda

Loading Model 1 (Water Detection)...


Using cache found in /root/.cache/torch/hub/facebookresearch_pytorchvideo_main


Model 1 loaded successfully!

Loading Model 2 (Pollution Detection)...
Model 2 (ConvNeXt) loaded successfully!


In [24]:
print("Please upload your drone video")
uploaded = files.upload()

if uploaded:
    video_filename = list(uploaded.keys())[0]
    INPUT_VIDEO_PATH = os.path.join("/content", video_filename)
    print(f"\n '{video_filename}' Successfully uploaded")
else:
    print("No vide uploaded")

Please upload your drone video


Saving Drone footage shows toxic foam floating on the Yamuna River in Delhi.mp4 to Drone footage shows toxic foam floating on the Yamuna River in Delhi.mp4

 'Drone footage shows toxic foam floating on the Yamuna River in Delhi.mp4' Successfully uploaded


In [25]:
import tensorflow as tf
from tensorflow.keras.applications.convnext import preprocess_input

def process_and_predict_video(video_path):
    cap = cv2.VideoCapture(video_path)
    frames = []
    raw_frames = []

    count = 0
    while cap.isOpened() and count < 100:
        ret, frame = cap.read()
        if not ret or frame is None:
            break
        raw_frames.append(frame.copy())
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame_rgb)
        count += 1
    cap.release()

    total_frames = len(frames)
    if total_frames == 0:
        print("Error: Invalid video file or failed to decode frames.")
        return

    num_frames = 16
    if total_frames < num_frames:
        while len(frames) < num_frames:
            frames.append(frames[-1])
            raw_frames.append(raw_frames[-1])
    else:
        indices = np.linspace(0, total_frames - 1, num_frames, dtype=int)
        frames = [frames[i] for i in indices]
        raw_frames = [raw_frames[i] for i in indices]

    processed_frames = []
    for frame in frames:
        img_tensor = torch.from_numpy(frame).permute(2, 0, 1).float() / 255.0
        img_tensor = water_transform(img_tensor)
        processed_frames.append(img_tensor)

    video_tensor = torch.stack(processed_frames, dim=1).unsqueeze(0).to(DEVICE)

    water_classes = ['No_Water', 'Water']
    with torch.no_grad():
        outputs = model_water(video_tensor)
        _, predicted = torch.max(outputs, 1)
        water_result = water_classes[predicted.item()]

    print(f"\n[Model 1 Execution] Result: {water_result}")

    if water_result == 'No_Water':
        print("\nstatus: WATER NOT DETECTED!")
        print(" Please upload a different video.")
    else:
        print("\nstatus: WATER DETECTED! Initiating localized zoom and frame capture...")

        output_folder = "/content/captured_water_frames"
        os.makedirs(output_folder, exist_ok=True)

        pollution_preds = []

        select_indices = [4, 8, 12]

        for i, idx in enumerate(select_indices):
            rf = raw_frames[idx]
            h, w, _ = rf.shape

            start_x, start_y = int(w * 0.25), int(h * 0.25)
            end_x, end_y = int(w * 0.75), int(h * 0.75)
            zoomed_frame = rf[start_y:end_y, start_x:end_x]

            frame_save_path = os.path.join(output_folder, f"water_frame_{i+1}.jpg")
            cv2.imwrite(frame_save_path, zoomed_frame)

            img_input = cv2.cvtColor(zoomed_frame, cv2.COLOR_BGR2RGB)
            img_input = cv2.resize(img_input, (224, 224))

            img_input = preprocess_input(img_input)
            img_input = np.expand_dims(img_input, axis=0)

            pred = model_pollution.predict(img_input, verbose=0)
            pollution_preds.append(pred[0])

        print(f"Success: 3 localized water frames captured and saved to '{output_folder}'")

        avg_pred = np.mean(pollution_preds, axis=0)
        final_class_idx = np.argmax(avg_pred)

        pollution_classes = ['algal_blooms', 'chemical', 'clean_water', 'foam', 'plastic', 'sediment']
        detected_status = pollution_classes[final_class_idx]


        if detected_status == 'clean_water':
            print(f"SURFACE WATER IS CLEAN")
            print(f"Classification Target: {detected_status.upper()}")
        else:
            print(f" POLLUTION DETECTED")
            print(f" Polution Type: {detected_status.upper()}")


process_and_predict_video(INPUT_VIDEO_PATH)


[Model 1 Execution] Result: Water

status: WATER DETECTED! Initiating localized zoom and frame capture...


Success: 3 localized water frames captured and saved to '/content/captured_water_frames'
 POLLUTION DETECTED
 Polution Type: FOAM
